# Generador de PLS con Llama 3.2 1B

Cuaderno que ajusta Llama-3.2-1B-Instruct con LoRA para generar resúmenes en lenguaje sencillo a partir de abstracts biomédicos. Cubre preparación de datos Cochrane, entrenamiento en 4 bits y evaluación rápida.

## Tabla de contenidos

- [1. Preparacion de archivos](#1-preparacion-de-archivos)

- [2. Verificacion de hardware](#2-verificacion-de-hardware)

- [3. Instalacion de dependencias](#3-instalacion-de-dependencias)

- [4. Login y seleccion de modelo](#4-login-y-seleccion-de-modelo)

- [5. Imports y semillas](#5-imports-y-semillas)

- [6. Inventario de versiones](#6-inventario-de-versiones)

- [7. Montaje de Drive y carpetas](#7-montaje-de-drive-y-carpetas)

- [8. Deteccion del repositorio y rutas de texto](#8-deteccion-del-repositorio-y-rutas-de-texto)

- [9. Emparejamiento de pares Cochrane](#9-emparejamiento-de-pares-cochrane)

- [10. Formato instruct y datasets](#10-formato-instruct-y-datasets)

- [11. Plantilla de chat y tokenizacion](#11-plantilla-de-chat-y-tokenizacion)

- [12. Fine-tuning LoRA en 4 bits](#12-fine-tuning-lora-en-4-bits)

- [13. Guardado e inferencia de prueba](#13-guardado-e-inferencia-de-prueba)

- [14. Evaluacion rapida (BERTScore y legibilidad)](#14-evaluacion-rapida-bertscore-y-legibilidad)

- [15. Evaluacion alternativa ampliada](#15-evaluacion-alternativa-ampliada)


## 1. Preparacion de archivos

Celda recordatorio para descomprimir el zip `bridging-the-gap-in-health-literacy-main` antes de procesar los textos.


In [ ]:
##!unzip bridging-the-gap-in-health-literacy-main.zip

Se truncaron las últimas líneas 5000 del resultado de transmisión.
  inflating: bridging-the-gap-in-health-literacy-main/data_collection_and_processing/web_scrapping/pls/10.1002-14651858.CD007633.pub2-pls.txt  
  inflating: bridging-the-gap-in-health-literacy-main/data_collection_and_processing/web_scrapping/pls/10.1002-14651858.CD007635.pub2-pls.txt  
  inflating: bridging-the-gap-in-health-literacy-main/data_collection_and_processing/web_scrapping/pls/10.1002-14651858.CD007639.pub2-pls.txt  
  inflating: bridging-the-gap-in-health-literacy-main/data_collection_and_processing/web_scrapping/pls/10.1002-14651858.CD007641.pub3-pls.txt  
  inflating: bridging-the-gap-in-health-literacy-main/data_collection_and_processing/web_scrapping/pls/10.1002-14651858.CD007644.pub3-pls.txt  
  inflating: bridging-the-gap-in-health-literacy-main/data_collection_and_processing/web_scrapping/pls/10.1002-14651858.CD007645.pub2-pls.txt  
  inflating: bridging-the-gap-in-health-literacy-main/data_collection

## 2. Verificacion de hardware

Consulta GPU disponible con `nvidia-smi` y revisa que PyTorch detecte CUDA.


In [ ]:
!nvidia-smi
import torch, sys, platform
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Sun Oct 26 19:06:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Instalacion de dependencias

Instala versiones recientes de Transformers, Accelerate, PEFT, bitsandbytes, Datasets y utilidades de evaluacion.


In [ ]:
!pip -q install "transformers>=4.44.0" "accelerate>=0.33.0" "peft>=0.12.0" \
                "bitsandbytes>=0.43.0" datasets sentencepiece "huggingface_hub>=0.25.0" \
                evaluate pandas bert-score textstat


## 4. Login y seleccion de modelo

Autentica en Hugging Face, verifica acceso al modelo `meta-llama/Llama-3.2-1B-Instruct` y usa `TinyLlama` como respaldo. Configura el tokenizer con plantilla de chat y padding.


In [ ]:
# === Modelo y login a Hugging Face (con verificación y fallback) ===
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer
import getpass, os

# 1) Define tu modelo objetivo (puedes cambiarlo más tarde si quieres)
PRIMARY_MODEL = "meta-llama/Llama-3.2-1B-Instruct"      # gated
FALLBACK_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # libre

# 2) Login si hace falta
if not os.environ.get("HUGGINGFACE_TOKEN"):
    os.environ["HUGGINGFACE_TOKEN"] = getpass.getpass("🔑 HUGGINGFACE_TOKEN (HF): ")
login(token=os.environ["HUGGINGFACE_TOKEN"], add_to_git_credential=True)

# 3) Verifica acceso al modelo primario
api = HfApi()
MODEL_NAME = PRIMARY_MODEL
try:
    _ = api.model_info(PRIMARY_MODEL)  # falla si no tienes acceso
    print(f"✅ Acceso confirmado a: {PRIMARY_MODEL}")
except Exception as e:
    print(f"⚠️ Sin acceso a {PRIMARY_MODEL}: {e}")
    print(f"→ Usando fallback: {FALLBACK_MODEL}")
    MODEL_NAME = FALLBACK_MODEL

# 4) Tokenizer (chat template) y padding
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
assert tokenizer.chat_template is not None, "El tokenizer debe tener chat_template."

print("Modelo en uso:", MODEL_NAME)

🔑 HUGGINGFACE_TOKEN (HF): ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Acceso confirmado a: meta-llama/Llama-3.2-1B-Instruct
Modelo en uso: meta-llama/Llama-3.2-1B-Instruct


## 5. Imports y semillas

Carga dependencias minimas para resumir con LoRA (sin TRL), fija semillas y activa ajustes ligeros de rendimiento.


In [ ]:
# ==== Imports mínimos para RESUMEN con LoRA en Colab (sin TRL) ====
import os, json, math, random, gc
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from datasets import Dataset  # usamos Dataset, no load_dataset
# login ya lo haces en la celda anterior

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from peft import LoraConfig, get_peft_model

# (Las siguientes se importan más adelante cuando se evalúa)
# from evaluate import load       # para BERTScore (lo haremos en la celda de evaluación)
# import textstat                 # legibilidad (también en celda de evaluación)

# Semilla y chequeo rápido de GPU + pequeños tweaks de perf
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Imports OK (resumen + LoRA, sin TRL)")

GPU: Tesla T4
Imports OK (resumen + LoRA, sin TRL)


## 6. Inventario de versiones

Imprime versiones de Python, CUDA, Transformers, Accelerate, PEFT, bitsandbytes, Datasets y bibliotecas relacionadas para dejar trazabilidad del entorno.


In [ ]:
import torch, transformers, accelerate, peft, bitsandbytes as bnb, datasets, huggingface_hub, sentencepiece
import pandas as pd, sys, platform

print("Python:", sys.version.split()[0], "| OS:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM (approx):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2), "GB")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("Datasets:", datasets.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Pandas:", pd.__version__)

# Opcional: activar TF32 para un pelín de perf
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

Python: 3.12.12 | OS: Linux-6.6.105+-x86_64-with-glibc2.35
CUDA available: True
GPU: Tesla T4
Total VRAM (approx): 15.83 GB
PyTorch: 2.8.0+cu126
Transformers: 4.57.1
Accelerate: 1.11.0
PEFT: 0.17.1
bitsandbytes: 0.48.1
Datasets: 4.0.0
HF Hub: 0.35.3
SentencePiece: 0.2.1
Pandas: 2.2.2


## 7. Montaje de Drive y carpetas

Monta Google Drive en Colab y define rutas base para datos y salidas del ajuste fino.


In [ ]:
from google.colab import drive
import os

MOUNT_POINT = "/content/drive"
if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)

BASE_DIR   = f"{MOUNT_POINT}/MyDrive/llama_finetune"
DATA_DIR   = f"{BASE_DIR}/data"
OUTPUT_DIR = f"{BASE_DIR}/outputs/llama-3.2-1b-instruct-lora"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 8. Deteccion del repositorio y rutas de texto

Localiza la carpeta extraida con las fuentes (`data_collection_and_processing/Data Sources`), confirma existencia de Cochrane y resume conteos de archivos por split.


In [ ]:
# Celda: Detectar repo y fijar BASE_TEXT_DIR (sin tocar OUTPUT_DIR ni montar Drive)
from pathlib import Path

# Detecta la carpeta del zip extraído en /content
CANDIDATES = [
    Path("/content/bridging-the-gap-in-health-literacy-main"),
    *Path("/content").glob("bridging-the-gap-in-health-literacy*"),
]
REPO_DIR = next((p for p in CANDIDATES if p.exists()), None)
assert REPO_DIR is not None, "No encontré la carpeta del zip en /content. ¿Seguro ya lo descomprimiste?"

# Directorio donde están los .txt
BASE_TEXT_DIR = REPO_DIR / "data_collection_and_processing" / "Data Sources"
assert BASE_TEXT_DIR.exists(), f"No existe {BASE_TEXT_DIR}. Revisa la ruta real."

# Reusar OUTPUT_DIR creado en la celda de Drive
assert 'OUTPUT_DIR' in globals(), "Primero ejecuta la celda de Drive que define OUTPUT_DIR."
print("REPO_DIR     :", REPO_DIR)
print("BASE_TEXT_DIR:", BASE_TEXT_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)

REPO_DIR     : /content/bridging-the-gap-in-health-literacy-main
BASE_TEXT_DIR: /content/bridging-the-gap-in-health-literacy-main/data_collection_and_processing/Data Sources
OUTPUT_DIR   : /content/drive/MyDrive/llama_finetune/outputs/llama-3.2-1b-instruct-lora


In [ ]:
from pathlib import Path

# Detecta la carpeta del repo extraído en /content
CANDIDATES = [
    Path("/content/bridging-the-gap-in-health-literacy-main"),
    *Path("/content").glob("bridging-the-gap-in-health-literacy*"),
]
REPO_DIR = next((p for p in CANDIDATES if p.exists()), None)
assert REPO_DIR is not None, "No encontré la carpeta del zip en /content. ¿Seguro ya lo descomprimiste?"

# Donde están los textos (Cochrane, etc.)
BASE_TEXT_DIR = REPO_DIR / "data_collection_and_processing" / "Data Sources"
assert BASE_TEXT_DIR.exists(), f"No existe {BASE_TEXT_DIR}. Revisa la ruta real."

# IMPORTANTE: no redefinimos OUTPUT_DIR aquí; usamos el de la celda de Drive
assert 'OUTPUT_DIR' in globals(), "OUTPUT_DIR debe venir de la celda de Drive."
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("REPO_DIR     :", REPO_DIR)
print("BASE_TEXT_DIR:", BASE_TEXT_DIR, "| exists:", BASE_TEXT_DIR.exists())
print("OUTPUT_DIR   :", OUTPUT_DIR)

# (Opcional) Rutas específicas de Cochrane y conteo rápido de .txt
COCHRANE_DIR = BASE_TEXT_DIR / "Cochrane"
print("COCHRANE_DIR :", COCHRANE_DIR, "| exists:", COCHRANE_DIR.exists())

def count_txts(d: Path):
    return sum(1 for p in d.rglob("*.txt")) if d.exists() else 0

if COCHRANE_DIR.exists():
    print("train/non_pls .txt:", count_txts(COCHRANE_DIR / "train" / "non_pls"))
    print("train/pls     .txt:", count_txts(COCHRANE_DIR / "train" / "pls"))
    print("test/non_pls  .txt:", count_txts(COCHRANE_DIR / "test" / "non_pls"))
    print("test/pls      .txt:", count_txts(COCHRANE_DIR / "test" / "pls"))

REPO_DIR     : /content/bridging-the-gap-in-health-literacy-main
BASE_TEXT_DIR: /content/bridging-the-gap-in-health-literacy-main/data_collection_and_processing/Data Sources | exists: True
OUTPUT_DIR   : /content/drive/MyDrive/llama_finetune/outputs/llama-3.2-1b-instruct-lora
COCHRANE_DIR : /content/bridging-the-gap-in-health-literacy-main/data_collection_and_processing/Data Sources/Cochrane | exists: True
train/non_pls .txt: 31118
train/pls     .txt: 16241
test/non_pls  .txt: 7664
test/pls      .txt: 3940


## 9. Emparejamiento de pares Cochrane

Normaliza nombres de archivos, alinea abstracts (`non_pls`) con versiones PLS y construye DataFrames de train y test mostrando ejemplos.


In [ ]:
# === Emparejador robusto Cochrane: non_pls (abstract) → pls (plain language) ===
from pathlib import Path
import re, pandas as pd

COCHRANE_DIR = Path(BASE_TEXT_DIR) / "Cochrane"
assert COCHRANE_DIR.exists(), f"No existe {COCHRANE_DIR}"

def read_txt(p: Path):
    try:
        t = p.read_text(encoding="utf-8", errors="ignore").strip()
        return t if t else None
    except Exception:
        return None

# Normaliza nombre de archivo → clave base común
# Ejemplos de entrada:
#   10.1002-14651858.CD000009.pub4-pls.txt
#   10.1002-14651858.CD000009.pub4-pls.txt_accumulated_section7.txt
#   10.1002-14651858.CD000006.pub2-abstract.txt
#   10.1002-14651858.CD000006.pub2-abstract.txt_accumulated_section13.txt
def base_key(name: str) -> str:
    s = name.lower()
    # 1) elimina cualquier sufijo de secciones acumuladas
    s = re.sub(r'(_accumulated_section\d+|_section\d+)\.txt$', '.txt', s)
    # 2) quita el sufijo de tipo
    s = re.sub(r'-(pls|abstract)\.txt$', '', s)
    # 3) quita extensión final
    s = re.sub(r'\.txt$', '', s)
    return s

# prioridad: prefiere el archivo "principal" sin sufijos de sección acumulada
def score_preferencia(p: Path) -> int:
    n = p.name.lower()
    # exacto "-pls.txt" o "-abstract.txt" (mejor)
    if re.search(r'-(pls|abstract)\.txt$', n):
        return 0
    # con _sectionNN (peor)
    if re.search(r'_section\d+\.txt$', n):
        return 2
    # con _accumulated_sectionNN (aún peor)
    if re.search(r'_accumulated_section\d+\.txt$', n):
        return 3
    # fallback
    return 1

def collect_pairs_split(split: str):
    d_non = COCHRANE_DIR / split / "non_pls"
    d_pls = COCHRANE_DIR / split / "pls"
    assert d_non.exists() and d_pls.exists(), f"Faltan carpetas en {split}: {d_non} / {d_pls}"

    non_files = sorted(d_non.rglob("*.txt"))
    pls_files = sorted(d_pls.rglob("*.txt"))

    # agrupa por clave base
    non_map = {}
    for p in non_files:
        k = base_key(p.name)
        non_map.setdefault(k, []).append(p)

    pls_map = {}
    for p in pls_files:
        k = base_key(p.name)
        pls_map.setdefault(k, []).append(p)

    common = sorted(set(non_map).intersection(pls_map))
    pairs = []
    for k in common:
        # elige mejor candidato de cada lado por score de preferencia
        src = sorted(non_map[k], key=score_preferencia)[0]
        tgt = sorted(pls_map[k], key=score_preferencia)[0]
        src_txt, tgt_txt = read_txt(src), read_txt(tgt)
        if src_txt and tgt_txt:
            pairs.append({
                "id": k,
                "input": src_txt,     # texto complejo (abstract) -> resumir
                "output": tgt_txt,    # PLS (referencia)
                "src_file": src.name,
                "tgt_file": tgt.name
            })
    print(f"[{split}] candidatos: non_pls={len(non_files)} | pls={len(pls_files)} | emparejados={len(pairs)}")
    return pairs

train_pairs = collect_pairs_split("train")
test_pairs  = collect_pairs_split("test")

df_train = pd.DataFrame(train_pairs)
df_eval  = pd.DataFrame(test_pairs)
print("train shape:", df_train.shape, "| eval shape:", df_eval.shape)
display(df_train.head(5)[["id","src_file","tgt_file"]])

[train] candidatos: non_pls=31118 | pls=16241 | emparejados=3578
[test] candidatos: non_pls=7664 | pls=3940 | emparejados=218
train shape: (3578, 5) | eval shape: (218, 5)


,id,src_file,tgt_file
0,10.1002-14651858.cd000006.pub2,10.1002-14651858.CD000006.pub2-abstract.txt,10.1002-14651858.CD000006.pub2-pls.txt
1,10.1002-14651858.cd000009.pub4,10.1002-14651858.CD000009.pub4-abstract.txt,10.1002-14651858.CD000009.pub4-pls.txt
2,10.1002-14651858.cd000012.pub4,10.1002-14651858.CD000012.pub4-abstract.txt,10.1002-14651858.CD000012.pub4-pls.txt
3,10.1002-14651858.cd000022.pub4,10.1002-14651858.CD000022.pub4-abstract.txt,10.1002-14651858.CD000022.pub4-pls.txt
4,10.1002-14651858.cd000024.pub5,10.1002-14651858.CD000024.pub5-abstract.txt,10.1002-14651858.CD000024.pub5-pls.txt


## 10. Formato instruct y datasets

Define la instruccion base y convierte los DataFrames de entrenamiento y evaluacion a objetos `Dataset` listos para supervisar resúmenes.


In [ ]:
from datasets import Dataset
import pandas as pd

INSTRUCTION = (
    "Resume el texto en lenguaje sencillo para público general. "
    "Usa frases cortas, evita jerga y conserva los puntos clave. "
    "Extensión objetivo: 5–7 frases."
)

def to_instruct(df: pd.DataFrame):
    return Dataset.from_pandas(pd.DataFrame({
        "instruction": INSTRUCTION,
        "input": df["input"],
        "output": df["output"],   # PLS referencia
    }), preserve_index=False)

train_ds = to_instruct(df_train)
eval_ds  = to_instruct(df_eval)
len(train_ds), len(eval_ds), train_ds[0]


(3578,
 218,
 {'instruction': 'Resume el texto en lenguaje sencillo para público general. Usa frases cortas, evita jerga y conserva los puntos clave. Extensión objetivo: 5–7 frases.',
  'input': "Background\nApproximately 70% of women will experience perineal trauma following vaginal delivery and will require stitches. This may result in pain, suture removal and superficial dyspareunia. \nObjectives\nTo assess the effects of different suture materials on short‐ and long‐term morbidity following perineal repair. \nSearch methods\nWe searched the Cochrane Pregnancy and Childbirth Group's Trials Register (February 2010). \nSelection criteria\nRandomised trials comparing different suture materials for perineal repair after vaginal delivery. \nData collection and analysis\nTwo review authors independently assessed trial quality and extracted data.\nMain results\nWe included 18 trials with 10,171 women; comparisons included: catgut with standard synthetic (nine trials), rapidly absorbing syn

## 11. Plantilla de chat y tokenizacion

Genera textos con template de chat que incluyen instruccion, entrada y respuesta; luego tokeniza en lotes, enmascarando los tokens del prompt para el calculo de perdida.


In [ ]:
from transformers import AutoTokenizer
# asumo que ya tienes tokenizer cargado y MODEL_NAME definido
assert tokenizer.chat_template is not None
MAX_LEN = 512  # baja a 768/512 si te falta VRAM

def build_with_answer(example):
    messages = [
        {"role":"system","content":"Eres un asistente que escribe resúmenes claros y sencillos para público general."},
        {"role":"user","content": f"{example['instruction']}\n\nTexto:\n{example['input']}"},
        {"role":"assistant","content": example["output"]},
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return example

train_with_ans = train_ds.map(build_with_answer)
eval_with_ans  = eval_ds.map(build_with_answer)

def tokenize_with_mask(batch):
    msgs_prompt = [
        [{"role":"system","content":"Eres un asistente que escribe resúmenes claros y sencillos para público general."},
         {"role":"user","content": f"{INSTRUCTION}\n\nTexto:\n{inp}"}]
        for inp in batch["input"]
    ]
    prompt_texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_prompt]
    full_texts   = batch["text"]

    enc_prompt = tokenizer(prompt_texts, truncation=True, max_length=MAX_LEN, padding="max_length")
    enc_full   = tokenizer(full_texts,   truncation=True, max_length=MAX_LEN, padding="max_length")

    input_ids  = enc_full["input_ids"]; attention = enc_full["attention_mask"]
    labels = []
    for i in range(len(input_ids)):
        lab = input_ids[i].copy()
        prompt_len = int(sum(enc_prompt["attention_mask"][i]))
        for j in range(prompt_len): lab[j] = -100  # ignora pérdida en el prompt
        labels.append(lab)
    return {"input_ids": input_ids, "attention_mask": attention, "labels": labels}

train_tok = train_with_ans.map(tokenize_with_mask, batched=True, remove_columns=train_with_ans.column_names)
eval_tok  = eval_with_ans.map(tokenize_with_mask,  batched=True, remove_columns=eval_with_ans.column_names)

print(len(train_tok), len(eval_tok))

Map:   0%|          | 0/3578 [00:00<?, ? examples/s]

Map:   0%|          | 0/218 [00:00<?, ? examples/s]

Map:   0%|          | 0/3578 [00:00<?, ? examples/s]

Map:   0%|          | 0/218 [00:00<?, ? examples/s]

3578 218


## 12. Fine-tuning LoRA en 4 bits

Carga el modelo cuantizado, aplica preparacion para k-bit training, agrega adaptadores LoRA y lanza `Trainer` con configuracion segura para VRAM limitada.


In [ ]:
import os, torch, gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

gc.collect(); torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# 1) Carga base en 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 2) MUY IMPORTANTE: preparar el modelo para k-bit training + GC
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

# 3) Aplica LoRA (puedes usar el set “ligero” para ahorrar VRAM)
peft_config = LoraConfig(
    r=8,                  # r pequeño = menos VRAM
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"],  # sin FFN para aligerar
)
model = get_peft_model(model, peft_config)
model.train()  # asegura modo train

# (Opcional) chequeo de parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# 4) Collator y argumentos
dcollator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,             # prueba 1 si quieres “smoke test”
    per_device_train_batch_size=1,  # seguro para Colab Pro
    gradient_accumulation_steps=8,  # efectivo 8
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    logging_strategy="steps",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",      # si da warning, quita esta línea
    max_grad_norm=1.0,
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,    # redundante con las líneas de arriba, pero OK
    group_by_length=True,           # menos padding
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=dcollator,
    tokenizer=tokenizer,            # el warning “deprecated” es informativo; puede quedarse
)

res = trainer.train()
print("Training loss:", res.training_loss)
print(trainer.evaluate())

/tmp/ipython-input-4215764015.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Trainable params: 1,703,936 / 750,979,072 (0.23%)


Step,Training Loss
10,2.892000
20,2.580600
30,2.039700
40,1.706500
50,1.597500
60,1.594500
70,1.559300
80,1.569700
90,1.559400
100,1.513700


Training loss: 1.4587729583893503


{'eval_loss': 1.3911874294281006, 'eval_runtime': 127.2398, 'eval_samples_per_second': 1.713, 'eval_steps_per_second': 0.22, 'epoch': 2.0}


## 13. Guardado e inferencia de prueba

Guarda pesos LoRA y tokenizer en `OUTPUT_DIR`, define la funcion de generacion y muestra un resumen de ejemplo sobre el conjunto de evaluacion.


In [ ]:
# Guarda LoRA + tokenizer
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import torch
def generate_summary(text, max_input_tokens=768, max_new_tokens=256, do_sample=False):
    msgs = [
        {"role":"system","content":"Eres un asistente que escribe resúmenes claros y sencillos para público general."},
        {"role":"user","content": f"{INSTRUCTION}\n\nTexto:\n{text}"},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens).to(model.device)
    with torch.inference_mode(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=do_sample,
            temperature=0.7 if do_sample else None, top_p=0.9 if do_sample else None,
            use_cache=False, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

print(generate_summary(df_eval.iloc[0]["input"])[:500])

system

Cutting Knowledge Date: December 2023
Today Date: 26 Oct 2025

Eres un asistente que escribe resúmenes claros y sencillos para público general.user

Resume el texto en lenguaje sencillo para público general. Usa frases cortas, evita jerga y conserva los puntos clave. Extensión objetivo: 5–7 frases.

Texto:
Background
Urinary schistosomiasis is caused by an intravascular infection with parasitic Schistosoma haematobium worms. The adult worms typically migrate to the venous plexus of the h


## 14. Evaluacion rapida (BERTScore y legibilidad)

Selecciona una muestra pequeña, genera resúmenes, calcula BERTScore con `roberta-base` y compara legibilidad (Flesch/FKGL) frente a las referencias.


In [ ]:
# === Evaluación compacta y rápida: generación + BERTScore + legibilidad ===
import random, numpy as np, torch, gc
from evaluate import load
import textstat

assert 'df_eval' in globals(), "df_eval no existe (asegúrate de haber creado los pares)."
assert 'generate_summary' in globals(), "Falta la función generate_summary."

# 1) Muestra y generación (parámetros ligeros para acelerar)
random.seed(42)
N = min(20, len(df_eval))   # ajusta a 50 si quieres más
idx = random.sample(range(len(df_eval)), N)
srcs = df_eval.iloc[idx]["input"].tolist()
refs = df_eval.iloc[idx]["output"].tolist()

preds = [
    generate_summary(t, max_input_tokens=512, max_new_tokens=160, do_sample=False)
    for t in srcs
]

# Limpia memoria antes del cómputo de BERTScore
gc.collect(); torch.cuda.empty_cache()

# 2) BERTScore (rápido): usa un modelo base + GPU + batch
bertscore = load("bertscore")
bs = bertscore.compute(
    predictions=preds,
    references=refs,
    lang="en",                 # usa "es" si tus textos están en español
    model_type="roberta-base", # más rápido que roberta-large
    device="cuda:0" if torch.cuda.is_available() else "cpu",
    batch_size=8,
    verbose=True
)
p_mean, r_mean, f1_mean = float(np.mean(bs["precision"])), float(np.mean(bs["recall"])), float(np.mean(bs["f1"]))

print("--- Resultados Promedio de la Evaluación ---")
print(f"Tamaño de la muestra de evaluación: {N} ejemplos\n")
print("Relevancia (BERTScore):")
print(f"  - Precisión Promedio: {p_mean:.4f}")
print(f"  - Recall Promedio:    {r_mean:.4f}")
print(f"  - F1 Promedio:        {f1_mean:.4f}\n")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 40090.03 seconds, 0.00 sentences/sec
--- Resultados Promedio de la Evaluación ---
Tamaño de la muestra de evaluación: 20 ejemplos

Relevancia (BERTScore):
  - Precisión Promedio: 0.8288
  - Recall Promedio:    0.8533
  - F1 Promedio:        0.8408



In [ ]:
# 3) Legibilidad (Flesch/FKGL)
def leg_stats(texts):
    fkgl = [textstat.flesch_kincaid_grade(t) for t in texts]   # menor = mejor
    fre  = [textstat.flesch_reading_ease(t) for t in texts]    # mayor = mejor
    return float(np.mean(fkgl)), float(np.mean(fre))

fkgl_pred, fre_pred = leg_stats(preds)
fkgl_ref,  fre_ref  = leg_stats(refs)

print("Legibilidad (Flesch-Kincaid Grade Level - menor es mejor):")
print(f"  - Resúmenes Generados: {fkgl_pred:.2f}")
print(f"  - Resúmenes de Referencia: {fkgl_ref:.2f}\n")
print("Legibilidad (Flesch Reading Ease - mayor es mejor):")
print(f"  - Resúmenes Generados: {fre_pred:.2f}")
print(f"  - Resúmenes de Referencia: {fre_ref:.2f}")

Legibilidad (Flesch-Kincaid Grade Level - menor es mejor):
  - Resúmenes Generados: 13.10
  - Resúmenes de Referencia: 13.26

Legibilidad (Flesch Reading Ease - mayor es mejor):
  - Resúmenes Generados: 29.09
  - Resúmenes de Referencia: 36.86


## 15. Evaluacion alternativa ampliada

Incluye una variante auto-contenida que instala dependencias de evaluacion, toma hasta 50 ejemplos, calcula BERTScore y estadisticas de legibilidad.


In [ ]:
%pip -q install evaluate bert-score textstat
import random, numpy as np
from evaluate import load
import textstat

random.seed(42)
N = min(50, len(df_eval))
idx = random.sample(range(len(df_eval)), N)
refs = df_eval.iloc[idx]["output"].tolist()
srcs = df_eval.iloc[idx]["input"].tolist()

preds = [generate_summary(t, max_input_tokens=768, max_new_tokens=256, do_sample=False) for t in srcs]

bertscore = load("bertscore")
# Cambia lang="en" a "es" si tus textos están en español.
bs = bertscore.compute(predictions=preds, references=refs, lang="en")
p_mean, r_mean, f1_mean = float(np.mean(bs["precision"])), float(np.mean(bs["recall"])), float(np.mean(bs["f1"]))

def leg_stats(texts):
    fkgl = [textstat.flesch_kincaid_grade(t) for t in texts]  # menor mejor
    fre  = [textstat.flesch_reading_ease(t) for t in texts]   # mayor mejor
    return float(np.mean(fkgl)), float(np.mean(fre))

fkgl_pred, fre_pred = leg_stats(preds)
fkgl_ref,  fre_ref  = leg_stats(refs)

print("--- Resultados Promedio de la Evaluación ---")
print(f"Tamaño de la muestra de evaluación: {N} ejemplos\n")
print("Relevancia (BERTScore):")
print(f"  - Precisión Promedio: {p_mean:.4f}")
print(f"  - Recall Promedio:    {r_mean:.4f}")
print(f"  - F1 Promedio:        {f1_mean:.4f}\n")
print("Legibilidad (Flesch-Kincaid Grade Level - menor es mejor):")
print(f"  - Resúmenes Generados: {fkgl_pred:.2f}")
print(f"  - Resúmenes de Referencia: {fkgl_ref:.2f}\n")
print("Legibilidad (Flesch Reading Ease - mayor es mejor):")
print(f"  - Resúmenes Generados: {fre_pred:.2f}")
print(f"  - Resúmenes de Referencia: {fre_ref:.2f}")

KeyboardInterrupt: 